# 02 — Data Ingestion (Bab 2.2)Notebook ini mengimplementasikan proses **Batch Ingestion** — memuat dataset CSV ke dalam Spark DataFrame menggunakan **explicit schema** (sesuai best practice).

## 2.1 Inisialisasi SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("02_DataIngestion") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.driver.memory", "1g") \
    .getOrCreate()

print(f"✅ Spark {spark.version} connected")

## 2.2 Definisi Explicit SchemaSesuai RULES: *"Always define explicit schema, avoid schema inference in large datasets"*

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, 
    IntegerType, StringType
)

# Schema eksplisit untuk retail_sales_dataset.csv
schema = StructType([
    StructField("Transaction_ID", IntegerType(), nullable=False),
    StructField("Date", StringType(), nullable=False),
    StructField("Customer_ID", StringType(), nullable=False),
    StructField("Gender", StringType(), nullable=False),
    StructField("Age", IntegerType(), nullable=False),
    StructField("Product_Category", StringType(), nullable=False),
    StructField("Quantity", IntegerType(), nullable=False),
    StructField("Price_per_Unit", IntegerType(), nullable=False),
    StructField("Total_Amount", IntegerType(), nullable=False)
])

print("✅ Schema didefinisikan:")
for field in schema.fields:
    print(f"   {field.name:20s} → {str(field.dataType):15s} (nullable={field.nullable})")

## 2.3 Batch Ingestion — Load CSV

In [ ]:
# Load CSV dengan explicit schema
df_raw = spark.read.csv(
    "/data/retail_sales_dataset.csv",
    header=True,
    schema=schema
)

# Verifikasi hasil ingestion
print(f"✅ Data berhasil di-load!")
print(f"   Jumlah baris   : {df_raw.count()}")
print(f"   Jumlah kolom   : {len(df_raw.columns)}")
print(f"   Partisi        : {df_raw.rdd.getNumPartitions()}")
print()
print("=== Schema ===")
df_raw.printSchema()

## 2.4 Preview Data

In [ ]:
# Tampilkan 10 baris pertama
df_raw.show(10, truncate=False)

## 2.5 Statistik Deskriptif Awal

In [ ]:
# Summary statistics untuk kolom numerik
df_raw.select("Age", "Quantity", "Price_per_Unit", "Total_Amount").summary().show()

## 2.6 Simpan DataFrame sebagai Temporary ViewDisimpan agar bisa diakses di notebook selanjutnya via Spark SQL.

In [ ]:
df_raw.createOrReplaceTempView("raw_transactions")
print("✅ Temporary view 'raw_transactions' dibuat")
print("   Bisa diakses dengan: spark.sql('SELECT * FROM raw_transactions')")

In [ ]:
# Jangan stop spark — akan dipakai di notebook berikutnya
print("📌 SparkSession tetap aktif untuk notebook selanjutnya")